# Notebook 03 – LSTM Experiments

This notebook trains and evaluates **four LSTM configurations** from the heaviest to the smallest:

| Variant  | Hidden units | Layers | Approx. params |
|----------|-------------|--------|---------------|
| heavy    | 128          | 2      | ~200 k        |
| medium   | 64           | 2      | ~54 k         |
| light    | 32           | 1      | ~4.4 k        |
| tiny     | 16           | 1      | ~1.1 k        |

We then compare them on:
- **Accuracy**: MAPE on the held-out test set
- **Efficiency**: trainable parameters & model file size
- **Latency**: average inference time per sample

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.train import train
from src.evaluate import compare_all_variants

%matplotlib inline

CSV_PATH    = '../data/raw/train.csv'
MODELS_DIR  = '../models'
RESULTS_DIR = '../results'
EPOCHS      = 20
BATCH_SIZE  = 256

## 1. Train All Variants

> **Note:** Training the *heavy* variant on CPU takes ~15–30 min. Use a GPU runtime or reduce `EPOCHS` for quick experimentation.

In [ ]:
histories = {}

for variant in ['heavy', 'medium', 'light', 'tiny']:
    print(f'\n{'='*60}')
    print(f'  Training variant: {variant.upper()}')
    print(f'{'='*60}')
    model, hist = train(
        csv_path=CSV_PATH,
        variant=variant,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        models_dir=MODELS_DIR,
        results_dir=RESULTS_DIR,
    )
    histories[variant] = hist

## 2. Training Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.flatten()

for ax, (variant, hist) in zip(axes, histories.items()):
    ax.plot(hist['epoch'], hist['train_mse'], label='Train MSE')
    ax.plot(hist['epoch'], hist['val_mse'],   label='Val MSE',   linestyle='--')
    ax.set_title(f'Variant: {variant}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend()

plt.suptitle('Training & Validation MSE by Variant', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/lstm_loss_curves.png', dpi=120)
plt.show()

## 3. Evaluate All Variants

In [ ]:
summary = compare_all_variants(
    csv_path=CSV_PATH,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
)
summary

## 4. Accuracy vs Efficiency Trade-off

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAPE vs Parameters
axes[0].scatter(summary['n_parameters'], summary['mape_pct'], s=80, color='steelblue')
for _, row in summary.iterrows():
    axes[0].annotate(row['variant'], (row['n_parameters'], row['mape_pct']),
                     textcoords='offset points', xytext=(5, 5))
axes[0].set_xlabel('Trainable Parameters')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Accuracy vs Parameters')

# MAPE vs File Size
axes[1].scatter(summary['file_size_kb'], summary['mape_pct'], s=80, color='darkorange')
for _, row in summary.iterrows():
    axes[1].annotate(row['variant'], (row['file_size_kb'], row['mape_pct']),
                     textcoords='offset points', xytext=(5, 5))
axes[1].set_xlabel('Model File Size (KB)')
axes[1].set_ylabel('MAPE (%)')
axes[1].set_title('Accuracy vs File Size')

# MAPE vs Latency
axes[2].scatter(summary['latency_ms'], summary['mape_pct'], s=80, color='seagreen')
for _, row in summary.iterrows():
    axes[2].annotate(row['variant'], (row['latency_ms'], row['mape_pct']),
                     textcoords='offset points', xytext=(5, 5))
axes[2].set_xlabel('Inference Latency (ms)')
axes[2].set_ylabel('MAPE (%)')
axes[2].set_title('Accuracy vs Latency')

plt.suptitle('Accuracy–Efficiency Trade-off', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/accuracy_efficiency_tradeoff.png', dpi=120)
plt.show()

## 5. Summary Table

In [ ]:
display(summary.sort_values('mape_pct'))